<a href="https://colab.research.google.com/github/Tonikaneprv/PDA/blob/main/%D0%92%D0%B5%D0%B4%D0%B5%D1%80%D0%BD%D0%B8%D0%BA%D0%BE%D0%B2%D0%B0_%22PDA_26_27_%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D0%BA%D0%B0_1_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDA 26/27 · Практика 1. PySpark, Dask, Polars

За выполнение можно получить 5 баллов, которые пойдут в счет зачета.

Полезное:
- ноутбук Лекции 3 с примерами;
- [PySpark API Reference](https://spark.apache.org/docs/latest/api/python/reference/index.html), [список функций `pyspark.sql.functions`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html);
- [Dask DataFrame API](https://docs.dask.org/en/stable/dataframe-api.html), [Polars API](https://docs.pola.rs/api/python/stable/reference/index.html).

## Подготовка

In [2]:
# Установка библиотек (~1 минута)
!pip install -q "pyspark==4.2.*" "polars>=1.40" "dask[dataframe]" gdown

In [ ]:
import os, re, subprocess

def java_major():
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except FileNotFoundError:
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0

if java_major() < 17:
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print("Java", java_major())

In [3]:
# Данные: BTS On-Time Performance, 2023–2024
import glob, gdown

FILE_IDS = ['1NQuold7jvGZfHikbvtj0QynbhM90pT8L', '1HNAGhPHE6BmamcI0D767-BFhF4BWs6Hh', '1ERgltJRuybLl7sIUUVckzpUVzyP1ttCq', '1XrUk6ZYwQLQTjtsMHKxzzERP4dCLMEao', '1qy95Qpd93VClYPethMtMDJdAMBl21fnW', '1js5pbc_xe6kP7W4_lp6HaeXR0oAw2ZDL', '1qb4_uKGy0KU_4jRzaauqeObaaCQMtdBZ', '1TuC71TgrJDgpOvQVKl9jn89fAr9El1Pa', '1nt7e9Pszf2-IB3GsKHNRHZ_SPzxS0E2L', '1V5tCiD9jRCss0KoJxk-r1zOCw3onwez3', '1WsKGiVBV2p-sgiDePT6HZKUezWPc2zUH']
N_FILES = len(FILE_IDS)

if len(glob.glob("T_ONTIME_REPORTING*.csv")) < N_FILES:
    for fid in FILE_IDS[:N_FILES]:
        gdown.download(id=fid, quiet=True)

FILES = sorted(glob.glob("T_ONTIME_REPORTING*.csv"))[:N_FILES]
CSV_GLOB = "T_ONTIME_REPORTING*.csv"
print(len(FILES), "файлов:", FILES[:3], "...")

11 файлов: ['T_ONTIME_REPORTING 10.csv', 'T_ONTIME_REPORTING 11.csv', 'T_ONTIME_REPORTING 2.csv'] ...


In [4]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

spark = (
    SparkSession.builder
    .master("local[*]")                          # локальный режим: все ядра этой машины
    .appName("PDA-practice2")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

In [5]:
schema = T.StructType([
    T.StructField("YEAR", T.IntegerType()),
    T.StructField("FL_DATE", T.StringType()),
    T.StructField("OP_UNIQUE_CARRIER", T.StringType()),
    T.StructField("ORIGIN_AIRPORT_ID", T.IntegerType()),
    T.StructField("ORIGIN", T.StringType()),
    T.StructField("ORIGIN_CITY_NAME", T.StringType()),
    T.StructField("DEST_AIRPORT_ID", T.IntegerType()),
    T.StructField("DEST", T.StringType()),
    T.StructField("DEST_CITY_NAME", T.StringType()),
    T.StructField("DEP_DELAY", T.DoubleType()),
    T.StructField("ARR_DELAY", T.DoubleType()),
    T.StructField("CANCELLED", T.DoubleType()),
    T.StructField("ACTUAL_ELAPSED_TIME", T.DoubleType()),
    T.StructField("DISTANCE", T.DoubleType()),
    T.StructField("CARRIER_DELAY", T.DoubleType()),
    T.StructField("WEATHER_DELAY", T.DoubleType()),
    T.StructField("NAS_DELAY", T.DoubleType()),
    T.StructField("SECURITY_DELAY", T.DoubleType()),
    T.StructField("LATE_AIRCRAFT_DELAY", T.DoubleType()),
])

import time, pandas as pd, polars as pl, dask.dataframe as dd

def timeit(action, label):
    t = time.perf_counter(); res = action(); print(f"{label}: {time.perf_counter() - t:.2f} с"); return res

## Задача 1

Объедините все CSV-файлы в одну таблицу, сгруппируйте по перевозчику (`OP_UNIQUE_CARRIER`) и посчитайте суммарное расстояние (`DISTANCE`). Сделайте это в **pandas**, **Dask**, **Polars** и **PySpark** и измерьте время каждого варианта вместе с чтением данных.

1. Убедитесь, что все четыре ответа совпадают.
2. Какой инструмент оказался быстрее всех и почему? Почему PySpark на этих данных не лидер?
3. Как изменится время pandas, если читать только две нужные колонки (`usecols`)?

В Spark и Polars можно читать сразу все файлы по шаблону `CSV_GLOB`; в pandas — использовать `pd.concat`.

In [6]:
import numpy as np


PD_DTYPES = {f.name: ("int64" if isinstance(f.dataType, T.IntegerType)
                      else "float64" if isinstance(f.dataType, T.DoubleType)
                      else "object")
             for f in schema.fields}
PL_SCHEMA = {f.name: (pl.Int64 if isinstance(f.dataType, T.IntegerType)
                      else pl.Float64 if isinstance(f.dataType, T.DoubleType)
                      else pl.Utf8)
             for f in schema.fields}
COLS = list(PD_DTYPES)

In [7]:

def run_pandas():
    df = pd.concat([pd.read_csv(f, usecols=COLS, dtype=PD_DTYPES) for f in FILES], ignore_index=True)
    return df.groupby("OP_UNIQUE_CARRIER")["DISTANCE"].sum()

def run_dask():
    ddf = dd.read_csv(CSV_GLOB, usecols=COLS, dtype=PD_DTYPES)
    return ddf.groupby("OP_UNIQUE_CARRIER")["DISTANCE"].sum().compute()

def run_polars():
    return (pl.scan_csv(CSV_GLOB, schema_overrides=PL_SCHEMA)
              .group_by("OP_UNIQUE_CARRIER")
              .agg(pl.col("DISTANCE").sum())
              .collect())

def run_spark():
    return (spark.read.csv(CSV_GLOB, header=True, schema=schema)
                 .groupBy("OP_UNIQUE_CARRIER")
                 .agg(F.sum("DISTANCE").alias("DISTANCE"))
                 .toPandas())

r_pd = timeit(run_pandas, "pandas ")
r_dd = timeit(run_dask,   "Dask   ")
r_pl = timeit(run_polars, "Polars ")
r_sp = timeit(run_spark,  "PySpark")

pandas : 21.19 с
Dask   : 19.78 с
Polars : 1.53 с
PySpark: 25.62 с


In [8]:

res = pd.DataFrame({
    "pandas":  r_pd,
    "dask":    r_dd,
    "polars":  r_pl.to_pandas().set_index("OP_UNIQUE_CARRIER")["DISTANCE"],
    "pyspark": r_sp.set_index("OP_UNIQUE_CARRIER")["DISTANCE"],
}).sort_values("pandas", ascending=False)
display(res)


same = all(np.allclose(res["pandas"], res[c], rtol=1e-9) for c in ["dask", "polars", "pyspark"])
print("Число перевозчиков:", len(res), "| пропуски:", res.isna().any().any(), "| все ответы совпадают:", same)

,pandas,dask,polars,pyspark
OP_UNIQUE_CARRIER,,,,
WN,945737680.0,945737680.0,945737680.0,945737680.0
DL,843797741.0,843797741.0,843797741.0,843797741.0
AA,830570860.0,830570860.0,830570860.0,830570860.0
UA,744829917.0,744829917.0,744829917.0,744829917.0
AS,310462644.0,310462644.0,310462644.0,310462644.0
B6,287615634.0,287615634.0,287615634.0,287615634.0
OO,278959207.0,278959207.0,278959207.0,278959207.0
NK,240036482.0,240036482.0,240036482.0,240036482.0
F9,158261925.0,158261925.0,158261925.0,158261925.0


Число перевозчиков: 15 | пропуски: False | все ответы совпадают: True


In [9]:

def run_pandas_usecols():
    df = pd.concat([pd.read_csv(f, usecols=["OP_UNIQUE_CARRIER", "DISTANCE"]) for f in FILES], ignore_index=True)
    return df.groupby("OP_UNIQUE_CARRIER")["DISTANCE"].sum()

r_pd2 = timeit(run_pandas_usecols, "pandas, usecols=2 колонки")
print("Ответ тот же:", np.allclose(r_pd2.sort_index(), r_pd.sort_index()))

pandas, usecols=2 колонки: 9.43 с
Ответ тот же: True


## Задача 2. Кэширование

Отберите задержанные рейсы (`ARR_DELAY > 15`) и посчитайте по ним три разных агрегата: число рейсов по месяцам, среднюю задержку по перевозчикам и топ-5 направлений. Измерьте общее время без кэширования и с `cache()` отобранного DataFrame.

In [10]:

def with_dates(df):
    d = F.coalesce(
        F.try_to_timestamp(F.col("FL_DATE"), F.lit("M/d/yyyy h:mm:ss a")),
        F.try_to_timestamp(F.col("FL_DATE"), F.lit("yyyy-MM-dd")),
    ).cast("date")
    return df.withColumn("FL_DATE", d).withColumn("MONTH", F.month("FL_DATE"))

flights = with_dates(spark.read.csv(CSV_GLOB, header=True, schema=schema))
flights.printSchema()
print("Строк с нераспознанной датой:", flights.filter(F.col("FL_DATE").isNull()).count())
def three_aggs(delayed):
    by_month   = delayed.groupBy("YEAR", "MONTH").count().orderBy("YEAR", "MONTH").collect()
    by_carrier = (delayed.groupBy("OP_UNIQUE_CARRIER")
                         .agg(F.avg("ARR_DELAY").alias("avg_arr_delay"))
                         .orderBy(F.desc("avg_arr_delay")).collect())
    top_routes = (delayed.groupBy("ORIGIN", "DEST").count()
                         .orderBy(F.desc("count")).limit(5).collect())
    return by_month, by_carrier, top_routes


delayed = flights.filter(F.col("ARR_DELAY") > 15)
t0 = time.perf_counter()
res_nocache = three_aggs(delayed)
t_nocache = time.perf_counter() - t0
print(f"Без cache(): {t_nocache:.2f} с")

delayed_c = flights.filter(F.col("ARR_DELAY") > 15).cache()
t0 = time.perf_counter()
res_cache = three_aggs(delayed_c)
t_cache = time.perf_counter() - t0
print(f"С cache():   {t_cache:.2f} с  (ускорение x{t_nocache / t_cache:.1f})")
print("Результаты совпадают:", res_nocache == res_cache)
by_month, by_carrier, top_routes = res_cache
print("Задержанные рейсы по месяцам:")
display(pd.DataFrame(by_month, columns=["YEAR", "MONTH", "count"]))
print("Средняя задержка прилета по перевозчикам (только задержанные рейсы):")
display(pd.DataFrame(by_carrier, columns=["OP_UNIQUE_CARRIER", "avg_arr_delay"]))
print("Топ-5 направлений по числу задержанных рейсов:")
display(pd.DataFrame(top_routes, columns=["ORIGIN", "DEST", "count"]))
timeit(lambda: three_aggs(delayed_c), "С заполненным кэшем, повторно")
delayed_c.unpersist()

root
 |-- YEAR: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double (nullable = true)
 |-- NAS_DELAY: double (nullable = true)
 |-- SECURITY_DELAY: double (nullable = true)
 |-- LATE_AIRCRAFT_DELAY: double (nullable = true)
 |-- MONTH: integer (nullable = true)

Строк с нераспознанной датой: 0
Без cache(): 52.82 с
С cache():   37.34 с  (ускорение x1.4)
Результаты совпадают: True
Задерж

,YEAR,MONTH,count
0,2023,1,112890
1,2023,2,89724
2,2023,3,131292
3,2023,4,121744
4,2023,5,100446
5,2023,6,147875
6,2023,7,162632
7,2023,8,124261
8,2023,11,73004
9,2023,12,85517


Средняя задержка прилета по перевозчикам (только задержанные рейсы):


,OP_UNIQUE_CARRIER,avg_arr_delay
0,AA,88.468721
1,B6,85.157182
2,G4,84.725798
3,OO,82.532209
4,F9,80.776156
5,9E,79.627091
6,OH,78.589397
7,UA,74.650344
8,DL,73.994538
9,NK,73.569665


Топ-5 направлений по числу задержанных рейсов:


,ORIGIN,DEST,count
0,LAS,LAX,2569
1,OGG,HNL,2563
2,LAX,SFO,2551
3,LAX,LAS,2410
4,DEN,LAS,2320


С заполненным кэшем, повторно: 5.12 с


DataFrame[YEAR: int, FL_DATE: date, OP_UNIQUE_CARRIER: string, ORIGIN_AIRPORT_ID: int, ORIGIN: string, ORIGIN_CITY_NAME: string, DEST_AIRPORT_ID: int, DEST: string, DEST_CITY_NAME: string, DEP_DELAY: double, ARR_DELAY: double, CANCELLED: double, ACTUAL_ELAPSED_TIME: double, DISTANCE: double, CARRIER_DELAY: double, WEATHER_DELAY: double, NAS_DELAY: double, SECURITY_DELAY: double, LATE_AIRCRAFT_DELAY: double, MONTH: int]

## Задача 3. Spark SQL

1. **SQL.** Для каждого месяца найдите 3 аэропорта вылета с наибольшей долей отмененных рейсов среди аэропортов с не менее чем 1000 вылетов в этом месяце.
2. **DataFrame API.** Решите ту же задачу без SQL и убедитесь, что ответы совпадают.
3. Для каждого перевозчика найдите день с максимальной средней задержкой прилета.
4. Посчитайте ежедневное число рейсов и его изменение относительно предыдущего дня (`lag`). В какой день был самый большой спад?

In [11]:
from pyspark.sql import Window

flights.createOrReplaceTempView("flights")

top_cancel_sql = spark.sql("""
    WITH stats AS (
        SELECT YEAR, MONTH, ORIGIN,
               COUNT(*)       AS n_flights,
               AVG(CANCELLED) AS cancel_rate
        FROM flights
        GROUP BY YEAR, MONTH, ORIGIN
        HAVING COUNT(*) >= 1000
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY YEAR, MONTH
                                     ORDER BY cancel_rate DESC, ORIGIN) AS rn
        FROM stats
    )
    SELECT YEAR, MONTH, rn, ORIGIN, n_flights, ROUND(cancel_rate, 4) AS cancel_rate
    FROM ranked
    WHERE rn <= 3
    ORDER BY YEAR, MONTH, rn
""")
top_cancel_sql.show(40, truncate=False)
w = Window.partitionBy("YEAR", "MONTH").orderBy(F.desc("cancel_rate"), "ORIGIN")
top_cancel_df = (
    flights.groupBy("YEAR", "MONTH", "ORIGIN")
           .agg(F.count("*").alias("n_flights"), F.avg("CANCELLED").alias("cancel_rate"))
           .filter(F.col("n_flights") >= 1000)
           .withColumn("rn", F.row_number().over(w))
           .filter(F.col("rn") <= 3)
           .select("YEAR", "MONTH", "rn", "ORIGIN", "n_flights", F.round("cancel_rate", 4).alias("cancel_rate"))
           .orderBy("YEAR", "MONTH", "rn")
)

diff1 = top_cancel_sql.exceptAll(top_cancel_df).count()
diff2 = top_cancel_df.exceptAll(top_cancel_sql).count()
print("SQL и DataFrame API совпадают:", diff1 == 0 and diff2 == 0, f"(строк: {top_cancel_df.count()})")

w_c = Window.partitionBy("OP_UNIQUE_CARRIER").orderBy(F.desc("avg_arr_delay"))
worst_day = (
    flights.filter(F.col("ARR_DELAY").isNotNull())
           .groupBy("OP_UNIQUE_CARRIER", "FL_DATE")
           .agg(F.avg("ARR_DELAY").alias("avg_arr_delay"), F.count("*").alias("n_flights"))
           .withColumn("rn", F.row_number().over(w_c))
           .filter("rn = 1").drop("rn")
           .withColumn("avg_arr_delay", F.round("avg_arr_delay", 1))
           .orderBy(F.desc("avg_arr_delay"))
)
worst_day.show(30, truncate=False)

w_d = Window.orderBy("FL_DATE")
daily = (
    flights.groupBy("FL_DATE").count().withColumnRenamed("count", "n_flights")
           .withColumn("prev_n", F.lag("n_flights").over(w_d))
           .withColumn("change", F.col("n_flights") - F.col("prev_n"))
           .withColumn("weekday", F.date_format("FL_DATE", "EEEE"))
           .orderBy("FL_DATE")
)
daily.show(10)

print("Дни с самым большим спадом числа рейсов:")
daily.orderBy("change").show(5)
biggest = daily.orderBy("change").first()
print(f"Самый большой спад: {biggest['FL_DATE']} ({biggest['weekday']}) — "
      f"{biggest['n_flights']} рейсов против {biggest['prev_n']} днем ранее, изменение {biggest['change']}")

+----+-----+---+------+---------+-----------+
|YEAR|MONTH|rn |ORIGIN|n_flights|cancel_rate|
+----+-----+---+------+---------+-----------+
|2023|1    |1  |DAL   |5986     |0.0525     |
|2023|1    |2  |BUR   |2385     |0.0516     |
|2023|1    |3  |MDW   |6498     |0.0445     |
|2023|2    |1  |PDX   |4263     |0.0507     |
|2023|2    |2  |DAL   |5444     |0.0485     |
|2023|2    |3  |MKE   |1857     |0.0458     |
|2023|3    |1  |ALB   |1041     |0.0384     |
|2023|3    |2  |SYR   |1125     |0.0356     |
|2023|3    |3  |ANC   |1292     |0.0263     |
|2023|4    |1  |FLL   |8071     |0.0846     |
|2023|4    |2  |EWR   |12022    |0.037      |
|2023|4    |3  |JFK   |11638    |0.0308     |
|2023|5    |1  |ORF   |1800     |0.0256     |
|2023|5    |2  |HNL   |5166     |0.0203     |
|2023|5    |3  |KOA   |1457     |0.0178     |
|2023|6    |1  |EWR   |11339    |0.0848     |
|2023|6    |2  |LGA   |13568    |0.0638     |
|2023|6    |3  |ORF   |1927     |0.0602     |
|2023|7    |1  |EWR   |11646    |0

## Задача 4. Parquet

1. Сохраните `flights` в Parquet с партиционированием по `MONTH`. Сравните размер на диске с исходными CSV.
2. Прочитайте получившийся каталог в **Polars** (`pl.scan_parquet(..., hive_partitioning=True)`) и в **Dask** (`dd.read_parquet`).
3. Посчитайте среднюю задержку прилета по перевозчикам за декабрь по CSV и по Parquet во всех трех инструментах. Во сколько раз быстрее работает Parquet? Найдите в `explain()` Polars и Spark признаки того, что читаются только нужные колонки и папки.

In [13]:
import os
PARQUET_DIR = "flights_parquet"

def dir_size(path):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs)

timeit(lambda: flights.write.mode("overwrite").partitionBy("MONTH").parquet(PARQUET_DIR), "Запись Parquet")

csv_mb = sum(os.path.getsize(f) for f in FILES) / 2**20
pq_mb  = dir_size(PARQUET_DIR) / 2**20
print(f"CSV: {csv_mb:,.0f} МБ | Parquet: {pq_mb:,.0f} МБ | сжатие x{csv_mb / pq_mb:.1f}")
print(sorted(os.listdir(PARQUET_DIR)))

pl_flights = pl.scan_parquet(f"{PARQUET_DIR}/**/*.parquet", hive_partitioning=True)
print(pl_flights.collect_schema())
print("Polars, строк:", pl_flights.select(pl.len()).collect().item())

dd_flights = dd.read_parquet(PARQUET_DIR)
print("Dask, партиций:", dd_flights.npartitions, "| колонки:", list(dd_flights.columns))

CAR, DEL = "OP_UNIQUE_CARRIER", "ARR_DELAY"


def pd_month(s):
    m = s.str.extract(r"^(?:(\d{1,2})/|\d{4}-(\d{2}))")
    return m[0].fillna(m[1]).astype(int)

pl_month = (pl.when(pl.col("FL_DATE").str.contains("/"))
              .then(pl.col("FL_DATE").str.extract(r"^(\d{1,2})/", 1))
              .otherwise(pl.col("FL_DATE").str.extract(r"^\d{4}-(\d{2})", 1))
              .cast(pl.Int32).alias("MONTH"))


def spark_csv():
    return (with_dates(spark.read.csv(CSV_GLOB, header=True, schema=schema))
            .filter(F.col("MONTH") == 12).groupBy(CAR).agg(F.avg(DEL).alias(DEL)).toPandas())
def spark_pq():
    return (spark.read.parquet(PARQUET_DIR)
            .filter(F.col("MONTH") == 12).groupBy(CAR).agg(F.avg(DEL).alias(DEL)).toPandas())


def polars_csv():
    return (pl.scan_csv(CSV_GLOB, schema_overrides=PL_SCHEMA).with_columns(pl_month)
              .filter(pl.col("MONTH") == 12).group_by(CAR).agg(pl.col(DEL).mean()).collect())
def polars_pq():
    return (pl.scan_parquet(f"{PARQUET_DIR}/**/*.parquet", hive_partitioning=True)
              .filter(pl.col("MONTH") == 12).group_by(CAR).agg(pl.col(DEL).mean()).collect())


def dask_csv():
    ddf = dd.read_csv(CSV_GLOB, usecols=["FL_DATE", CAR, DEL], dtype=PD_DTYPES)
    ddf = ddf.assign(MONTH=ddf["FL_DATE"].map_partitions(pd_month, meta=("FL_DATE", "int64")))
    return ddf[ddf["MONTH"] == 12].groupby(CAR)[DEL].mean().compute()
def dask_pq():
    ddf = dd.read_parquet(PARQUET_DIR, columns=[CAR, DEL], filters=[("MONTH", "==", 12)])
    return ddf.groupby(CAR)[DEL].mean().compute()

times, answers = {}, {}
for name, f_csv, f_pq in [("Spark", spark_csv, spark_pq), ("Polars", polars_csv, polars_pq), ("Dask", dask_csv, dask_pq)]:
    t = time.perf_counter(); a_csv = f_csv(); t_csv = time.perf_counter() - t
    t = time.perf_counter(); a_pq  = f_pq();  t_pq  = time.perf_counter() - t
    times[name] = {"CSV, с": round(t_csv, 2), "Parquet, с": round(t_pq, 2), "ускорение": round(t_csv / t_pq, 1)}
    answers[name] = (a_csv, a_pq)

display(pd.DataFrame(times).T)

def to_series(a):
    if isinstance(a, pl.DataFrame): a = a.to_pandas()
    if isinstance(a, pd.DataFrame): a = a.set_index(CAR)[DEL]
    return a.sort_index()

dec = pd.DataFrame({f"{k} {fmt}": to_series(v[i]) for k, v in answers.items() for i, fmt in enumerate(["csv", "pq"])})
display(dec.round(2).sort_values("Spark pq", ascending=False))
print("Все ответы совпадают:", all(np.allclose(dec.iloc[:, 0], dec[c]) for c in dec.columns))

print("=== Spark ===")
(spark.read.parquet(PARQUET_DIR)
      .filter(F.col("MONTH") == 12).groupBy(CAR).agg(F.avg(DEL))
      .explain())

print("=== Polars ===")
print(pl.scan_parquet(f"{PARQUET_DIR}/**/*.parquet", hive_partitioning=True)
        .filter(pl.col("MONTH") == 12).group_by(CAR).agg(pl.col(DEL).mean())
        .explain())

Запись Parquet: 58.16 с
CSV: 733 МБ | Parquet: 61 МБ | сжатие x11.9
['._SUCCESS.crc', 'MONTH=1', 'MONTH=11', 'MONTH=12', 'MONTH=2', 'MONTH=3', 'MONTH=4', 'MONTH=5', 'MONTH=6', 'MONTH=7', 'MONTH=8', '_SUCCESS']
Schema({'YEAR': Int32, 'FL_DATE': Date, 'OP_UNIQUE_CARRIER': String, 'ORIGIN_AIRPORT_ID': Int32, 'ORIGIN': String, 'ORIGIN_CITY_NAME': String, 'DEST_AIRPORT_ID': Int32, 'DEST': String, 'DEST_CITY_NAME': String, 'DEP_DELAY': Float64, 'ARR_DELAY': Float64, 'CANCELLED': Float64, 'ACTUAL_ELAPSED_TIME': Float64, 'DISTANCE': Float64, 'CARRIER_DELAY': Float64, 'WEATHER_DELAY': Float64, 'NAS_DELAY': Float64, 'SECURITY_DELAY': Float64, 'LATE_AIRCRAFT_DELAY': Float64, 'MONTH': Int64})
Polars, строк: 6226864
Dask, партиций: 11 | колонки: ['YEAR', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'DEST_AIRPORT_ID', 'DEST', 'DEST_CITY_NAME', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED', 'ACTUAL_ELAPSED_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',

,"CSV, с","Parquet, с",ускорение
Spark,26.21,1.53,17.1
Polars,2.74,0.07,40.6
Dask,26.55,0.11,249.8


,Spark csv,Spark pq,Polars csv,Polars pq,Dask csv,Dask pq
OP_UNIQUE_CARRIER,,,,,,
B6,11.90,11.90,11.90,11.90,11.90,11.90
F9,7.38,7.38,7.38,7.38,7.38,7.38
NK,6.94,6.94,6.94,6.94,6.94,6.94
G4,5.19,5.19,5.19,5.19,5.19,5.19
HA,4.23,4.23,4.23,4.23,4.23,4.23
AA,4.06,4.06,4.06,4.06,4.06,4.06
OO,2.72,2.72,2.72,2.72,2.72,2.72
WN,1.88,1.88,1.88,1.88,1.88,1.88
OH,0.01,0.01,0.01,0.01,0.01,0.01


Все ответы совпадают: True
=== Spark ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[OP_UNIQUE_CARRIER#4309], functions=[avg(ARR_DELAY#4317)])
   +- Exchange hashpartitioning(OP_UNIQUE_CARRIER#4309, 8), ENSURE_REQUIREMENTS, [plan_id=4330]
      +- HashAggregate(keys=[OP_UNIQUE_CARRIER#4309], functions=[partial_avg(ARR_DELAY#4317)])
         +- Project [OP_UNIQUE_CARRIER#4309, ARR_DELAY#4317]
            +- FileScan parquet [OP_UNIQUE_CARRIER#4309,ARR_DELAY#4317,MONTH#4326] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/flights_parquet], PartitionFilters: [isnotnull(MONTH#4326), (MONTH#4326 = 12)], PushedFilters: [], ReadSchema: struct<OP_UNIQUE_CARRIER:string,ARR_DELAY:double>


=== Polars ===
AGGREGATE[maintain_order: false]
  [col("ARR_DELAY").mean()] BY [col("OP_UNIQUE_CARRIER")]
  FROM
  simple π 2/2 ["OP_UNIQUE_CARRIER", ... 1 other column]
    Parquet SCAN [flights_parquet/MONTH=12/part-00005-

## Задача 5. Партиции в Dask

1. Прочитайте CSV в Dask с `blocksize` 16 МБ, 64 МБ и 256 МБ. Сколько получается партиций?
2. Для каждого варианта измерьте время подсчета доли отмененных рейсов по перевозчикам. Какой размер партиции оптимален и почему слишком мелкие и слишком крупные партиции плохи?

In [14]:
blocks = ["16MB", "64MB", "256MB"]
part_res = {}
for bs in blocks:
    ddf = dd.read_csv(CSV_GLOB, blocksize=bs, usecols=COLS, dtype=PD_DTYPES)
    n = ddf.npartitions
    t = time.perf_counter()
    cancel_rate = ddf.groupby("OP_UNIQUE_CARRIER")["CANCELLED"].mean().compute()
    dt = time.perf_counter() - t
    part_res[bs] = {"партиций": n, "время, с": round(dt, 2)}
    print(f"blocksize={bs:>6}: {n:4d} партиций, {dt:.2f} с")

display(pd.DataFrame(part_res).T)
print("Доля отмененных рейсов по перевозчикам:")
display(cancel_rate.sort_values(ascending=False).round(4).to_frame("cancel_rate"))
sizes = pd.Series({os.path.basename(f): os.path.getsize(f) / 2**20 for f in FILES}).round(0)
print(f"Файлов: {len(sizes)}, средний размер {sizes.mean():.0f} МБ, суммарно {sizes.sum():.0f} МБ")
print("Ожидаемое число партиций:", {b: int(sum(np.ceil(sizes / int(b[:-2])))) for b in blocks})

blocksize=  16MB:   43 партиций, 17.37 с
blocksize=  64MB:   11 партиций, 19.77 с
blocksize= 256MB:   11 партиций, 18.94 с


,партиций,"время, с"
16MB,43.0,17.37
64MB,11.0,19.77
256MB,11.0,18.94


Доля отмененных рейсов по перевозчикам:


,cancel_rate
OP_UNIQUE_CARRIER,
YX,0.0279
F9,0.0229
9E,0.0227
AS,0.0224
B6,0.0213
UA,0.0213
OH,0.0183
NK,0.0171
OO,0.0165


Файлов: 11, средний размер 66 МБ, суммарно 731 МБ
Ожидаемое число партиций: {'16MB': 52, '64MB': 19, '256MB': 11}


## Завершение сессии

In [15]:
spark.stop()